# z304 — Etapa 4: Entrenamiento Final y Submit

Input  : `z302_features.parquet`, `z302_inferencia.parquet`, `z303_hiperparametros.json`
Output : `z304_predicciones.csv` → Kaggle

Responsabilidades:
- Leer hiperparámetros del disco (sin re-correr Optuna)
- Entrenar LGBM con features categóricas declaradas
- Predecir sobre inferencia (201912 → 202002)
- Submit a Kaggle

## 0. Ambiente

In [ ]:
import os, shutil, subprocess

# El bucket ya está montado por gcsfuse en /home/ds/buckets/b1.
# SQLite de Optuna va al disco LOCAL (/home/ds), no al bucket (gcsfuse no soporta locks).
BASE       = '/home/ds/buckets/b1'
LOCAL_HOME = '/home/ds'

os.makedirs(f'{BASE}/exp',      exist_ok=True)
os.makedirs(f'{BASE}/datasets', exist_ok=True)

# Kaggle auth: usar el de ~/.kaggle si ya existe; si no, buscarlo en el bucket
kaggle_dst = os.path.expanduser('~/.kaggle/kaggle.json')
os.makedirs(os.path.dirname(kaggle_dst), exist_ok=True)
if os.path.exists(kaggle_dst):
    os.chmod(kaggle_dst, 0o600)
    print('Kaggle auth OK (ya estaba en ~/.kaggle)')
else:
    _encontrado = False
    for cand in [f'{BASE}/kaggle.json', f'{BASE}/kaggle/kaggle.json']:
        if os.path.exists(cand):
            shutil.copy(cand, kaggle_dst)
            os.chmod(kaggle_dst, 0o600)
            print(f'Kaggle auth OK (copiado de {cand})')
            _encontrado = True
            break
    if not _encontrado:
        print('⚠️  kaggle.json no encontrado. Subilo a ~/.kaggle/kaggle.json o al bucket.')

def descargar(archivo):
    url = f'https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{archivo}'
    dst = f'{BASE}/datasets/{archivo}'
    if not os.path.exists(dst):
        subprocess.run(['wget', url, '-O', dst], check=True)
    print(f'✅ {archivo}')

descargar('sell-in.txt.gz')
descargar('tb_productos.txt')
descargar('tb_stocks.txt')
descargar('product_id_apredecir201912.txt')

In [ ]:
!pip install -q uv
!uv pip install -q pyarrow lightgbm

## 1. Parámetros — palancas

In [ ]:
import os

PARAM = {
    'experimento': 'z304',
    'kaggle_competition': 'labo-iii-2026-rosario',

    # ── PALANCA 1 (debe coincidir con z301/z302/z303) ──────────────────
    'modo_agrupacion': 'producto',

    'path_apredecir': '/home/ds/buckets/b1/datasets/product_id_apredecir201912.txt',

    # ── PALANCA 12: sampling de filas para entrenamiento ────────────────
    'sampling_frac': None,

    # ── PALANCA 13: des-escalado ──────────────────────────────
    'desescalar': False,

    # ── PALANCA 14: clip de negativos ──────────────────────────
    'clip_min': 0.0,

    # ── PALANCA 16: peso por recencia (debe coincidir con z303) ────────────
    'decay_recencia': None,

    # ── PALANCA 17: qué experimento de hiperparámetros usar ───────────────
    # nombre del 'experimento' de z303 cuyo JSON se quiere cargar
    'experimento_hiper': 'z303_tweedie',

    # ── PALANCA 18: ensemble de semillas ─────────────────────────
    # lista de semillas: entrena un modelo por cada una y promedia las predicciones.
    # [102191] = un solo modelo (sin ensemble)
    'semillas_ensemble': [102191],

    'semilla': 102191,
}

# Paths derivados del modo → cada modo usa su dataset, su inferencia y su JSON
MODO = PARAM['modo_agrupacion']
PARAM['path_train']  = f'/home/ds/buckets/b1/exp/z302_features_{MODO}.parquet'
PARAM['path_infer']  = f'/home/ds/buckets/b1/exp/z302_inferencia_{MODO}.parquet'
PARAM['path_hiper']  = f"/home/ds/buckets/b1/exp/z303_hiper_{MODO}_{PARAM['experimento_hiper']}.json"
PARAM['path_output'] = f'/home/ds/buckets/b1/exp/z304_predicciones_{MODO}.csv'

ruta_exp = '/home/ds/buckets/b1/exp/' + PARAM['experimento']
os.makedirs(ruta_exp, exist_ok=True)
os.chdir(ruta_exp)
print('Parámetros:', PARAM)
print('Train:', PARAM['path_train'])
print('Hiper:', PARAM['path_hiper'])

## 2. Cargar hiperparámetros y datasets

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
import lightgbm as lgb
import json

with open(PARAM['path_hiper']) as f:
    cfg = json.load(f)

FEATURES     = cfg['features']
CAT_FEATURES = cfg.get('cat_features', [])
TIPO_TARGET  = cfg.get('tipo_target', 'nivel')
TARGET_COL   = 'target_delta' if TIPO_TARGET == 'delta' else 'target_nivel'
hiper        = cfg['hiperparametros']

print(f'Experimento Optuna: {cfg["experimento"]}')
print(f'Métrica: {cfg["metrica"]} = {cfg["mejor_valor"]:.4f}')
print(f'Tipo de target: {TIPO_TARGET}')
print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Categóricas: {CAT_FEATURES}')

df_train = pl.read_parquet(PARAM['path_train'])
df_infer = pl.read_parquet(PARAM['path_infer'])
tb_pred  = pl.read_csv(PARAM['path_apredecir'], separator='\t',
                       dtypes={'product_id': pl.Int32})

# Modo de agrupación propagado desde z301 (define si hay que sumar por producto)
MODO = df_infer['modo_agrupacion'][0] if 'modo_agrupacion' in df_infer.columns else 'producto'
print(f'Modo de agrupación: {MODO}')

print(f'\nTrain: {df_train.shape}  |  Inferencia: {df_infer.shape}')

## 3. Preparación X / y (con categóricas)

In [ ]:
df_train_pd = df_train.to_pandas()
df_infer_pd = df_infer.to_pandas()

if PARAM['sampling_frac'] is not None:
    df_train_pd = df_train_pd.sample(frac=PARAM['sampling_frac'], random_state=PARAM['semilla'])
    print(f'Sampling: {len(df_train_pd):,} filas')

# Verificar features
faltantes = [f for f in FEATURES if f not in df_train_pd.columns]
if faltantes:
    print(f'⚠️  Features faltantes: {faltantes}')
    FEATURES = [f for f in FEATURES if f in df_train_pd.columns]

# Declarar categóricas como dtype category (consistente train/infer)
for c in CAT_FEATURES:
    if c in df_train_pd.columns:
        df_train_pd[c] = df_train_pd[c].astype('category')
        df_infer_pd[c] = df_infer_pd[c].astype('category')

X_train = df_train_pd[FEATURES]
y_train = df_train_pd[TARGET_COL].values
X_infer = df_infer_pd[FEATURES]

print(f'X_train: {X_train.shape}  |  X_infer: {X_infer.shape}')

## 4. Entrenamiento final

In [ ]:
OBJECTIVE_LGBM = cfg.get('objective_lgbm', 'regression')

def calcular_pesos(periodos_serie, decay):
    """Peso por recencia: el período más reciente pesa 1, cada mes hacia atrás decae."""
    if decay is None:
        return None
    periodos = sorted(periodos_serie.unique())
    idx = {p: i for i, p in enumerate(periodos)}
    n = len(periodos)
    return periodos_serie.map(lambda p: decay ** (n - 1 - idx[p])).values

w_train = calcular_pesos(df_train_pd['periodo'], PARAM['decay_recencia'])
if w_train is not None:
    print(f'Sample weights por recencia (decay={PARAM["decay_recencia"]})')

# Ensemble de semillas: entrena un modelo por semilla, promedia predicciones
semillas = PARAM['semillas_ensemble']
print(f'Objective: {OBJECTIVE_LGBM}  |  Semillas: {semillas}')

modelos = []
for s in semillas:
    params_lgbm = {
        'objective':     OBJECTIVE_LGBM,
        'metric':        'mae',
        'verbosity':     -1,
        'boosting_type': 'gbdt',
        'seed':          s,
        **hiper
    }
    m = lgb.LGBMRegressor(**params_lgbm)
    m.fit(X_train, y_train, sample_weight=w_train, categorical_feature=CAT_FEATURES)
    modelos.append(m)
    print(f'  ✅ modelo semilla {s} entrenado (n_estimators={m.n_estimators_})')

print(f'Ensemble de {len(modelos)} modelo(s) listo.')

## 5. Feature importance

In [ ]:
fi = pd.Series(modelos[0].feature_importances_, index=FEATURES).sort_values(ascending=False)
print('Feature importance (gain, primer modelo del ensemble):')
print(fi.to_string())

## 6. Predicción

In [ ]:
# Promediar predicciones de todos los modelos del ensemble
preds = np.column_stack([m.predict(X_infer) for m in modelos])
y_pred = preds.mean(axis=1)

# Reconstruir nivel si el target es delta: nivel = tn_actual + delta
if TIPO_TARGET == 'delta':
    tn_actual = df_infer_pd['tn'].values
    y_pred = tn_actual + y_pred
    print('Target delta: nivel reconstruido = tn_actual + delta')
else:
    print('Target nivel: predicción directa')

if PARAM['desescalar']:
    if 'media_rolling' in df_infer_pd.columns:
        media = df_infer_pd['media_rolling'].values
        y_pred = y_pred * np.where(media > 0, media, 1.0)
        print('Des-escalado aplicado.')
    else:
        print('⚠️  desescalar=True pero media_rolling no encontrada.')

y_pred = np.maximum(y_pred, PARAM['clip_min'])

print(f'min={y_pred.min():.3f}  max={y_pred.max():.3f}  mean={y_pred.mean():.3f}')
print(f'Predicciones = 0: {(y_pred == 0).sum()}')

## 7. Armar tabla de submit

In [ ]:
df_pred = pl.DataFrame({
    'product_id': df_infer_pd['product_id'].values.astype('int32'),
    'periodo':    df_infer_pd['periodo'].values.astype('int32'),
    'tn':         y_pred.astype('float64'),
})

ultimo_p = int(df_infer_pd['periodo'].max())
df_pred_p = df_pred.filter(pl.col('periodo') == ultimo_p)

# Kaggle mide a nivel product_id. Si el modo es cliente_producto,
# sumar las predicciones de todos los clientes de cada producto.
if MODO == 'cliente_producto':
    df_pred_final = (
        df_pred_p
        .group_by('product_id')
        .agg(pl.col('tn').sum())
    )
    print(f'Modo cliente_producto: agregado a nivel producto.')
else:
    df_pred_final = df_pred_p.drop('periodo')

print(f'Período usado: {ultimo_p}  |  Productos: {df_pred_final.height}')

# Completar con 0 los productos sin fila
tb_base = tb_pred.with_columns(pl.lit(0.0).alias('tn'))
tb_submit = (
    tb_base
    .join(df_pred_final, on='product_id', how='left', suffix='_pred')
    .with_columns(pl.coalesce(['tn_pred', 'tn']).alias('tn'))
    .select(['product_id', 'tn'])
    .sort('product_id')
)
print(f'Submit: {tb_submit.height} productos')
print(tb_submit.describe())

## 8. Guardar y submit

In [ ]:
tb_submit.write_csv(PARAM['path_output'])
print(f'✅ Guardado: {PARAM["path_output"]}')

import subprocess
mensaje = f"LGBM z304 | {cfg['metrica']}={cfg['mejor_valor']:.4f}"
res = subprocess.run(
    ['kaggle', 'competitions', 'submit',
     '-c', PARAM['kaggle_competition'],
     '-f', PARAM['path_output'],
     '-m', mensaje],
    capture_output=True, text=True
)
print('stdout:', res.stdout)
print('stderr:', res.stderr)
print('returncode:', res.returncode)

## 9. Diagnóstico (WAPE in-sample, orientativo)

In [ ]:
# Diagnóstico in-sample con el promedio del ensemble
preds_tr = np.column_stack([m.predict(X_train) for m in modelos])
y_train_pred = preds_tr.mean(axis=1)

if TIPO_TARGET == 'delta':
    tn_tr      = df_train_pd['tn'].values
    pred_nivel = np.maximum(tn_tr + y_train_pred, 0.0)
    real_nivel = tn_tr + y_train
else:
    pred_nivel = np.maximum(y_train_pred, 0.0)
    real_nivel = y_train

wape_train = np.abs(real_nivel - pred_nivel).sum() / real_nivel.sum()
print(f'WAPE in-sample sobre nivel (NO es el de Kaggle): {wape_train:.4f}  ({wape_train*100:.2f}%)')